# Correlation Breakdown During Market Stress

## Research question
Do relationships between major asset classes change when implied market volatility is elevated?

This notebook compares daily return correlations across normal- and high-VIX environments.

**Assets**
- **SPY:** US equities
- **AGG:** US investment-grade aggregate bonds
- **GLD:** Gold
- **TLT:** Long-duration US Treasuries

The VIX is used to classify volatility regimes. It is **not** included in the asset correlation matrices.

## 1. Download market data

Download adjusted daily prices for the four ETFs and the VIX index. The sample begins in 2019.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

tickers = ['SPY', 'AGG', 'GLD', 'TLT', '^VIX']

raw_data = yf.download(
    tickers,
    start='2019-01-01',
    end='2026-11-10',
    auto_adjust=True,
    progress=False
)

prices = raw_data['Close'].copy()
vix = prices.pop('^VIX')

prices.head()

## 2. Prepare returns and align observations

Calculate daily percentage returns without forward-filling missing prices. Keep dates for which all four ETF returns and the VIX observation are available.

This complete-case approach ensures that the regime labels and asset returns refer to the same dates.

In [ ]:
print("Missing ETF prices:")
print(prices.isna().sum())

print("\\nMissing VIX observations:")
print(vix.isna().sum())

returns = prices.pct_change(fill_method=None).dropna(how='any')

vix = vix.reindex(returns.index)
valid_dates = vix.notna()

returns = returns.loc[valid_dates]
vix = vix.loc[valid_dates]

print(f"\\nAligned observations: {len(returns):,}")
print(f"Date range: {returns.index.min().date()} to {returns.index.max().date()}")

## 3. Define volatility regimes

Classify a day as **high VIX** when the VIX level is above its 80th percentile over the full sample. All remaining observations are classified as **normal VIX**.

This is a retrospective, descriptive classification. Because the threshold uses the full sample, it should not be interpreted as a real-time trading signal.

In [ ]:
vix_threshold = vix.quantile(0.80)

regime = pd.Series(
    np.where(vix > vix_threshold, 'high', 'normal'),
    index=vix.index,
    name='regime'
)

data = returns.join(regime)

print(f"80th percentile VIX threshold: {vix_threshold:.2f}")
print("\\nRegime observations:")
print(data['regime'].value_counts())

print("\\nRegime proportions:")
print(data['regime'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

## 4. Compare correlations across regimes

Calculate a Pearson correlation matrix for ETF daily returns separately for normal- and high-VIX observations.

The difference matrix is defined as:

**High-VIX correlation − Normal-VIX correlation**

- A positive difference means the pair's correlation is higher in the high-VIX regime.
- A negative difference means the pair's correlation is lower in the high-VIX regime.

A lower correlation does not automatically mean better diversification; interpretation depends on the starting correlation and the portfolio context.

In [ ]:
normal_returns = data.loc[data['regime'] == 'normal', returns.columns]
high_returns = data.loc[data['regime'] == 'high', returns.columns]

normal_corr = normal_returns.corr()
high_corr = high_returns.corr()
corr_diff = high_corr - normal_corr

n_normal = len(normal_returns)
n_high = len(high_returns)

print(f"Normal-VIX observations: {n_normal:,}")
print(f"High-VIX observations: {n_high:,}")

display(normal_corr.round(2))
display(high_corr.round(2))
display(corr_diff.round(2))

## 5. Visualise the correlation matrices

The first two heatmaps use the same colour scale (−1 to +1), making them directly comparable. The third heatmap shows the change in correlation, with its own symmetric scale around zero.

In [ ]:
sns.set_theme(style='white', font_scale=0.95)

delta_limit = np.nanmax(np.abs(corr_diff.to_numpy()))
delta_limit = max(0.05, np.ceil(delta_limit / 0.05) * 0.05)

cmap = 'RdBu_r'

fig = plt.figure(figsize=(19, 7))
gs = fig.add_gridspec(
    1, 5,
    width_ratios=[1, 1, 0.045, 1, 0.045],
    wspace=0.45
)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
cax1 = fig.add_subplot(gs[0, 2])
ax3 = fig.add_subplot(gs[0, 3])
cax2 = fig.add_subplot(gs[0, 4])

sns.heatmap(
    normal_corr, ax=ax1, annot=True, fmt='.2f',
    cmap=cmap, vmin=-1, vmax=1, center=0,
    square=True, linewidths=0.5, cbar=False
)
ax1.set_title(f'Normal VIX (n = {n_normal:,})', fontsize=13, pad=15)

sns.heatmap(
    high_corr, ax=ax2, annot=True, fmt='.2f',
    cmap=cmap, vmin=-1, vmax=1, center=0,
    square=True, linewidths=0.5, cbar=False
)
ax2.set_title(f'High VIX (n = {n_high:,})', fontsize=13, pad=15)

sns.heatmap(
    corr_diff, ax=ax3, annot=True, fmt='.2f',
    cmap=cmap, vmin=-delta_limit, vmax=delta_limit,
    center=0, square=True, linewidths=0.5, cbar=False
)
ax3.set_title(
    'Change in Correlation\\n(High VIX − Normal VIX)',
    fontsize=13, pad=15
)

corr_map = mpl.cm.ScalarMappable(
    norm=mpl.colors.Normalize(vmin=-1, vmax=1),
    cmap=cmap
)
fig.colorbar(corr_map, cax=cax1, label='Correlation')

diff_map = mpl.cm.ScalarMappable(
    norm=mpl.colors.Normalize(vmin=-delta_limit, vmax=delta_limit),
    cmap=cmap
)
fig.colorbar(diff_map, cax=cax2, label='Change in correlation')

fig.suptitle(
    'How Do Asset Correlations Change During High-Volatility Periods?',
    fontsize=17, fontweight='bold', y=0.98
)
fig.text(
    0.5, 0.92,
    'Daily ETF returns | High VIX defined as VIX above its historical 80th percentile',
    ha='center', fontsize=10
)

for ax in [ax1, ax2, ax3]:
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.subplots_adjust(left=0.04, right=0.97, bottom=0.08, top=0.84)
plt.savefig(
    'vix_conditional_correlation_analysis.png',
    dpi=300, bbox_inches='tight', facecolor='white'
)
plt.show()

## 6. Summarise the pairwise changes

The correlation matrices are symmetric and their diagonals are all 1. To avoid reporting each pair twice, list only the six unique asset pairs.

In [ ]:
from itertools import combinations

pair_results = pd.DataFrame([
    {
        'Asset pair': f'{asset1}–{asset2}',
        'Normal VIX': normal_corr.loc[asset1, asset2],
        'High VIX': high_corr.loc[asset1, asset2],
        'Change': corr_diff.loc[asset1, asset2]
    }
    for asset1, asset2 in combinations(returns.columns, 2)
])

display(
    pair_results.style.format({
        'Normal VIX': '{:.2f}',
        'High VIX': '{:.2f}',
        'Change': '{:+.2f}'
    })
)

## 7. Sensitivity check: VIX threshold

Check whether the SPY–TLT result changes materially when the high-VIX threshold is moved from the 80th percentile to the 75th or 85th percentile.

This is a simple sensitivity check, not a formal statistical significance test.

In [ ]:
sensitivity_results = []

for percentile in [0.75, 0.80, 0.85]:
    high_vix = vix > vix.quantile(percentile)
    normal_vix = ~high_vix

    sensitivity_results.append({
        'VIX percentile': percentile,
        'Normal correlation': returns.loc[normal_vix, 'SPY'].corr(
            returns.loc[normal_vix, 'TLT']
        ),
        'High-VIX correlation': returns.loc[high_vix, 'SPY'].corr(
            returns.loc[high_vix, 'TLT']
        ),
        'Normal observations': int(normal_vix.sum()),
        'High-VIX observations': int(high_vix.sum())
    })

sensitivity = pd.DataFrame(sensitivity_results)

display(
    sensitivity.style.format({
        'VIX percentile': '{:.0%}',
        'Normal correlation': '{:.2f}',
        'High-VIX correlation': '{:.2f}'
    })
)

## 8. Findings and limitations

Use the results above to record the main observations. Avoid assuming that all correlations decline during high-volatility periods; the direction and size of the change vary by asset pair.

**Points to consider when interpreting the results:**
- Correlation describes the strength and direction of a linear relationship; it does not establish causation.
- The high-VIX sample is smaller than the normal-VIX sample, so its estimates may be less precise.
- The 80th-percentile threshold is estimated using the full historical sample.
- These are correlations of daily returns. They do not, by themselves, quantify portfolio losses or prove that diversification benefits improved or deteriorated.
- The analysis is descriptive. Statistical uncertainty and portfolio-level risk effects would require further work.